In [1]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, DataCollatorWithPadding
import torch
from torch.utils.data import DataLoader
import random
from tqdm import tqdm

/workspace/llm-autoformalization/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROMPT = """
Please autoformalize the following problem in Lean 4.
{nl_problem}
"""

def prepare_dataset(dataset, size=None):
    if size is None:
        size = len(dataset)

    data = []
    for sample in dataset:
        nl_problem = sample["natural_language_statement"]
        formal_problem = sample["formal_statement"]

        prompt = PROMPT.format(nl_problem=nl_problem)
        completion = f"```lean\n{formal_problem}\n```"

        data.append({
            "prompt": [{"role": "user", "content": prompt}],
            "completion": [{"role": "assistant", "content": completion}]
        })

    data = data[:size]
    
    return Dataset.from_list(data)

In [3]:
dsname = "internlm/Lean-Workbook"
ds = load_dataset(dsname)

In [4]:
dataset = prepare_dataset(ds["train"])
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [5]:
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda"
)


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 520.48it/s]


In [6]:
config = SFTConfig(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=len(train_dataset)*0.05,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="epoch"
)

In [7]:
train_dataset[0]

{'prompt': [{'role': 'user',
   'content': '\nPlease autoformalize the following problem in Lean 4.\nIf $(a,b) =1$ there exists such $n$ that $an \\equiv 1$ $(mod b)$\n'}],
 'completion': [{'role': 'assistant',
   'content': '```lean\ntheorem lean_workbook_plus_23365 {a b : ℕ} (hab : Nat.Coprime a b) : ∃ n, a*n ≡ 1 [ZMOD b]   :=  by sorry\n```'}]}

In [8]:
train = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=config
)

Tokenizing eval dataset: 100%|██████████| 5043/5043 [00:03<00:00, 1265.45 examples/s]


In [9]:
train.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
500,0.289786,0.289839
1000,0.272885,0.266594
1500,0.256118,0.247628
2000,0.221730,0.233181
2500,0.209745,0.223071
3000,0.209182,0.217623
3500,0.218517,0.214221
4000,0.196310,0.213047
4500,0.209629,0.212927
5000,0.208998,0.212909


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.43s/it]


TrainOutput(global_step=5043, training_loss=0.25953293365546215, metrics={'train_runtime': 2121.766, 'train_samples_per_second': 9.507, 'train_steps_per_second': 2.377, 'total_flos': 8.740540673728512e+16, 'train_loss': 0.25953293365546215})

In [13]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

In [11]:
eval_dataset = load_dataset("PAug/ProofNetVerif")["test"]

In [12]:
def get_prompt(sample):
    messages = [
        {"role": "system", "content": "You are an expert in mathematics and Lean 4."},
        {"role": "user", "content": PROMPT.format(nl_problem=sample["nl_statement"])}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return {"prompt": text}

eval_dataset = eval_dataset.map(get_prompt)

In [14]:
def tokenize_fn(example):
    return tokenizer(example["prompt"], truncation=True, max_length=512)

eval_dataset = eval_dataset.map(tokenize_fn)
eval_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

In [15]:
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

loader = DataLoader(
    eval_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collator,
)

In [16]:
all_answers = []

device = "cuda" if torch.cuda.is_available() else "cpu"
for batch in tqdm(loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256
        )

    outputs = outputs.cpu()

    texts = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    all_answers.extend(texts)

100%|██████████| 91/91 [06:35<00:00,  4.35s/it]


In [23]:
eval_dataset = eval_dataset.add_column(
    "answers",
    all_answers
)

In [25]:
eval_dataset.to_csv("benchmark_pred.csv", index=False)

Creating CSV from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  4.12ba/s]


3033654